In [0]:
customers_df = spark.read.format("delta") \
.load("dbfs:/FileStore/bronze/customers")

products_df = spark.read.format("delta") \
.load("dbfs:/FileStore/bronze/products")

orders_df = spark.read.format("delta") \
.load("dbfs:/FileStore/bronze/orders")

preferences_df = spark.read.format("delta") \
.load("dbfs:/FileStore/bronze/preferences")

In [0]:
customers_df = customers_df.na.fill("Unknown")
products_df = products_df.na.fill("Unknown")
orders_df = orders_df.na.fill("Unknown")

In [0]:
from pyspark.sql.functions import *

preferences_flat = preferences_df.select(
    "customer_id",
    "preferred_channel",
    col("loyalty.tier").alias("tier"),
    col("loyalty.points").alias("points")
)

display(preferences_flat)

customer_id,preferred_channel,tier,points
C101,Online,Gold,1200
C102,Store,Silver,700
C104,Online,Platinum,2200
C108,Mobile App,Gold,1500


In [0]:
customer_pref_df = customers_df.join(
    preferences_flat,
    "customer_id",
    "left"
)

display(customer_pref_df)

customer_id,customer_name,city,state,customer_type,preferred_channel,tier,points
C101,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200
C102,Priya Reddy,Bangalore,Karnataka,Regular,Store,Silver,700
C103,Amit Kumar,Mumbai,Maharashtra,Regular,null,null,null
C104,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200
C105,Farhan Ali,Chennai,Tamil Nadu,Regular,null,null,null
C106,Neha Singh,Pune,Maharashtra,Premium,null,null,null
C107,Arjun Verma,Hyderabad,Telangana,Regular,null,null,null
C108,Meera Nair,Kochi,Kerala,Premium,Mobile App,Gold,1500


In [0]:
retail_df = orders_df.join(
    customer_pref_df,
    "customer_id",
    "left"
).join(
    products_df,
    "product_id",
    "left"
)

display(retail_df)

product_id,customer_id,order_id,order_date,quantity,status,customer_name,city,state,customer_type,preferred_channel,tier,points,product_name,category,unit_price
P101,C101,O1001,2026-06-01,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Laptop,Electronics,65000
P102,C102,O1002,2026-06-01,2,Completed,Priya Reddy,Bangalore,Karnataka,Regular,Store,Silver,700,Mobile,Electronics,25000
P103,C103,O1003,2026-06-02,3,Pending,Amit Kumar,Mumbai,Maharashtra,Regular,null,null,null,Chair,Furniture,7000
P104,C104,O1004,2026-06-02,2,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Table,Furniture,12000
P105,C105,O1005,2026-06-03,4,Cancelled,Farhan Ali,Chennai,Tamil Nadu,Regular,null,null,null,Shoes,Fashion,4500
P106,C106,O1006,2026-06-03,1,Completed,Neha Singh,Pune,Maharashtra,Premium,null,null,null,Watch,Fashion,8000
P107,C107,O1007,2026-06-04,1,Completed,Arjun Verma,Hyderabad,Telangana,Regular,null,null,null,TV,Electronics,45000
P108,C108,O1008,2026-06-04,5,Completed,Meera Nair,Kochi,Kerala,Premium,Mobile App,Gold,1500,Bag,Fashion,3000
P102,C101,O1009,2026-06-05,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Mobile,Electronics,25000
P101,C104,O1010,2026-06-05,1,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Laptop,Electronics,65000


In [0]:
retail_df = retail_df.withColumn(
    "revenue",
    col("quantity").cast("int") *
    col("unit_price").cast("double")
)

In [0]:
retail_df = retail_df.withColumn(
    "order_month",
    date_format("order_date","yyyy-MM")
)

In [0]:
retail_df = retail_df.withColumn(
    "customer_segment",
    when(col("customer_type")=="Premium","High Value")
    .otherwise("Standard Value")
)

In [0]:
display(retail_df)

product_id,customer_id,order_id,order_date,quantity,status,customer_name,city,state,customer_type,preferred_channel,tier,points,product_name,category,unit_price,revenue,order_month,customer_segment
P101,C101,O1001,2026-06-01,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Laptop,Electronics,65000,65000.0,2026-06,High Value
P102,C102,O1002,2026-06-01,2,Completed,Priya Reddy,Bangalore,Karnataka,Regular,Store,Silver,700,Mobile,Electronics,25000,50000.0,2026-06,Standard Value
P103,C103,O1003,2026-06-02,3,Pending,Amit Kumar,Mumbai,Maharashtra,Regular,null,null,null,Chair,Furniture,7000,21000.0,2026-06,Standard Value
P104,C104,O1004,2026-06-02,2,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Table,Furniture,12000,24000.0,2026-06,High Value
P105,C105,O1005,2026-06-03,4,Cancelled,Farhan Ali,Chennai,Tamil Nadu,Regular,null,null,null,Shoes,Fashion,4500,18000.0,2026-06,Standard Value
P106,C106,O1006,2026-06-03,1,Completed,Neha Singh,Pune,Maharashtra,Premium,null,null,null,Watch,Fashion,8000,8000.0,2026-06,High Value
P107,C107,O1007,2026-06-04,1,Completed,Arjun Verma,Hyderabad,Telangana,Regular,null,null,null,TV,Electronics,45000,45000.0,2026-06,Standard Value
P108,C108,O1008,2026-06-04,5,Completed,Meera Nair,Kochi,Kerala,Premium,Mobile App,Gold,1500,Bag,Fashion,3000,15000.0,2026-06,High Value
P102,C101,O1009,2026-06-05,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Mobile,Electronics,25000,25000.0,2026-06,High Value
P101,C104,O1010,2026-06-05,1,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Laptop,Electronics,65000,65000.0,2026-06,High Value


In [0]:
retail_df.write.format("delta") \
.mode("overwrite") \
.save("dbfs:/FileStore/silver/retail")

In [0]:
silver_df = spark.read.format("delta") \
.load("dbfs:/FileStore/silver/retail")

display(silver_df)

product_id,customer_id,order_id,order_date,quantity,status,customer_name,city,state,customer_type,preferred_channel,tier,points,product_name,category,unit_price,revenue,order_month,customer_segment
P101,C101,O1001,2026-06-01,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Laptop,Electronics,65000,65000.0,2026-06,High Value
P102,C102,O1002,2026-06-01,2,Completed,Priya Reddy,Bangalore,Karnataka,Regular,Store,Silver,700,Mobile,Electronics,25000,50000.0,2026-06,Standard Value
P103,C103,O1003,2026-06-02,3,Pending,Amit Kumar,Mumbai,Maharashtra,Regular,null,null,null,Chair,Furniture,7000,21000.0,2026-06,Standard Value
P104,C104,O1004,2026-06-02,2,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Table,Furniture,12000,24000.0,2026-06,High Value
P105,C105,O1005,2026-06-03,4,Cancelled,Farhan Ali,Chennai,Tamil Nadu,Regular,null,null,null,Shoes,Fashion,4500,18000.0,2026-06,Standard Value
P106,C106,O1006,2026-06-03,1,Completed,Neha Singh,Pune,Maharashtra,Premium,null,null,null,Watch,Fashion,8000,8000.0,2026-06,High Value
P107,C107,O1007,2026-06-04,1,Completed,Arjun Verma,Hyderabad,Telangana,Regular,null,null,null,TV,Electronics,45000,45000.0,2026-06,Standard Value
P108,C108,O1008,2026-06-04,5,Completed,Meera Nair,Kochi,Kerala,Premium,Mobile App,Gold,1500,Bag,Fashion,3000,15000.0,2026-06,High Value
P102,C101,O1009,2026-06-05,1,Completed,Rahul Sharma,Hyderabad,Telangana,Premium,Online,Gold,1200,Mobile,Electronics,25000,25000.0,2026-06,High Value
P101,C104,O1010,2026-06-05,1,Completed,Sneha Patel,Delhi,Delhi,Premium,Online,Platinum,2200,Laptop,Electronics,65000,65000.0,2026-06,High Value
